In [1]:
import os
import random
import numpy as np
import soundfile as sf
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
from torchaudio.transforms import MelSpectrogram, AmplitudeToDB
from tqdm import tqdm
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

BASE_DIR    = "/kaggle/input/competitions/digitrecognition-ee708"
TRAIN_CSV   = os.path.join(BASE_DIR, "train.csv")
TRAIN_AUDIO = os.path.join(BASE_DIR, "train_audio", "train_audio")  # <-- nested
TEST_AUDIO  = os.path.join(BASE_DIR, "test_audio", "test_audio")    # <-- nested
OUTPUT_DIR  = "/kaggle/working"
TEST_CSV    = os.path.join(OUTPUT_DIR, "test.csv")

SR          = 16000
MAX_SAMPLES = 16000      # 1 second
N_MELS      = 64
N_FFT       = 400
HOP         = 160

BATCH_SIZE  = 128
EPOCHS      = 50
LR          = 1e-3
N_FOLDS     = 5
TTA_SHIFTS = [0, 800, -800, 400, -400, 1200, -1200, 200, -200, 600, -600]

print("Config ready.")

Using device: cuda
Config ready.


In [2]:
# Build test.csv
files   = sorted([f for f in os.listdir(TEST_AUDIO) if f.endswith('.wav')])
ids     = [os.path.splitext(f)[0] for f in files]
test_df = pd.DataFrame({'id': ids})
test_df.to_csv(TEST_CSV, index=False)
print(f"Built test.csv with {len(test_df)} test files")
print(f"Sample IDs: {ids[:3]}")

# Verify train
train_df = pd.read_csv(TRAIN_CSV)
print(f"Train samples: {len(train_df)}")

# Verify audio loading
import soundfile as sf
sample_path = os.path.join(TRAIN_AUDIO, train_df.iloc[0]['id'] + '.wav')
print(f"Trying to load: {sample_path}")
data, sr = sf.read(sample_path, dtype='float32')
print(f"Audio OK — shape: {data.shape}, SR: {sr}")

Built test.csv with 16200 test files
Sample IDs: ['test_000001', 'test_000002', 'test_000003']
Train samples: 37800
Trying to load: /kaggle/input/competitions/digitrecognition-ee708/train_audio/train_audio/train_000001.wav
Audio OK — shape: (40001,), SR: 48000


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
# remove
def pad_or_crop(wav):
    if wav.shape[-1] > MAX_SAMPLES:
        return wav[..., :MAX_SAMPLES]
    return F.pad(wav, (0, MAX_SAMPLES - wav.shape[-1]))

In [4]:
class DigitDataset(Dataset):
    def __init__(self, ids, train=True):
        self.ids   = ids
        self.train = train

    def __len__(self):
        return len(self.ids)

    def augment_wave(self, wav):
        wav = wav.clone()
        if random.random() < 0.6:
            wav = wav + random.uniform(0.001, 0.015) * torch.randn_like(wav)
        if random.random() < 0.6:
            shift = random.randint(-3200, 3200)
            wav   = torch.roll(wav, shifts=shift, dims=-1)
            if shift > 0:
                wav[..., :shift] = 0
            else:
                wav[..., shift:] = 0
        if random.random() < 0.5:
            wav = wav * random.uniform(0.7, 1.3)
        return wav

    def __getitem__(self, idx):
        aid = self.ids[idx]
        wav = all_wavs[aid].clone()
        if self.train:
            wav = self.augment_wave(wav)
        return wav, all_labels[aid]


In [5]:
class InceptionBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.b1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1), nn.GELU())
        self.b3 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.GELU())
        self.b5 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 5, padding=2), nn.GELU())
        self.b4 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1), nn.GELU())
        self.bn = nn.BatchNorm2d(out_ch * 4)

    def forward(self, x):
        x = torch.cat([self.b1(x), self.b3(x),
                        self.b5(x), self.b4(x)], dim=1)
        return F.relu(self.bn(x))

class DigitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.GELU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU(), nn.MaxPool2d(2)
        )
        self.inc1     = InceptionBlock(128, 64)   # output 256
        self.inc2     = InceptionBlock(256, 64)   # output 256
        self.mid_pool = nn.MaxPool2d(2)
        self.inc3     = InceptionBlock(256, 64)   # output 256
        self.inc4     = InceptionBlock(256, 64)   # output 256  ← extra block
        self.fc       = nn.Sequential(
            nn.Linear(256, 256), nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128), nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.inc1(x)
        x = self.inc2(x)
        x = self.mid_pool(x)
        x = self.inc3(x)
        x = self.inc4(x)
        x = x.mean(dim=[2, 3])
        return self.fc(x)


In [6]:
import pickle
from pathlib import Path

CACHE_DIR = "/kaggle/working/spec_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def precompute_specs(df, audio_dir, cache_dir, desc="Caching"):
    mel = MelSpectrogram(sample_rate=SR, n_fft=N_FFT,
                         hop_length=HOP, n_mels=N_MELS)
    db  = AmplitudeToDB()
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        cache_path = os.path.join(cache_dir, f"{row.id}.pt")
        if os.path.exists(cache_path):
            continue
        path = os.path.join(audio_dir, f"{row.id}.wav")
        data, sr = sf.read(path, dtype='float32')
        if data.ndim > 1:
            data = data.mean(axis=1)
        wav = torch.tensor(data).unsqueeze(0)
        if sr != SR:
            wav = torchaudio.functional.resample(wav, sr, SR)
        wav  = pad_or_crop(wav)
        spec = db(mel(wav))
        spec = (spec - spec.mean()) / (spec.std() + 1e-6)
        torch.save(spec, cache_path)

# Pre-compute all training specs (do this once)
print("Pre-computing spectrograms...")
train_df_full = pd.read_csv(TRAIN_CSV)
precompute_specs(train_df_full, TRAIN_AUDIO, CACHE_DIR, desc="Train specs")
print("Done caching!")


Pre-computing spectrograms...


Train specs: 100%|██████████| 37800/37800 [10:18<00:00, 61.11it/s]

Done caching!


In [7]:
# These live on GPU permanently
mel_gpu = MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT,
    hop_length=HOP, n_mels=N_MELS
).to(DEVICE)

db_gpu = AmplitudeToDB().to(DEVICE)

def wav_to_spec_gpu(wav_batch):
    # wav_batch: (B, 1, T) on GPU
    spec = db_gpu(mel_gpu(wav_batch))             # (B, 1, N_MELS, T)
    mean = spec.mean(dim=[2,3], keepdim=True)
    std  = spec.std(dim=[2,3], keepdim=True)
    spec = (spec - mean) / (std + 1e-6)
    return spec

def spec_augment_gpu(spec):
    # spec: (B, 1, F, T) on GPU
    B, C, F, T = spec.shape
    for _ in range(2):
        if random.random() < 0.5:
            t  = random.randint(5, 20)
            t0 = random.randint(0, max(1, T - t))
            spec[:, :, :, t0:t0+t] = 0
    for _ in range(2):
        if random.random() < 0.5:
            f  = random.randint(4, 15)
            f0 = random.randint(0, max(1, F - f))
            spec[:, :, f0:f0+f, :] = 0
    return spec

print("GPU mel transform ready on:", next(mel_gpu.parameters()).device 
      if len(list(mel_gpu.parameters())) > 0 else DEVICE)


GPU mel transform ready on: cuda


In [8]:
def train_epoch(model, loader, optimizer, loss_fn, train=True):
    model.train()
    total = 0
    for wav, y in loader:
        wav = wav.to(DEVICE, non_blocking=True)
        y   = y.to(DEVICE,   non_blocking=True)

        with torch.no_grad():
            spec = wav_to_spec_gpu(wav)
        if train:
            spec = spec_augment_gpu(spec)

        # Mixup
        if random.random() < 0.5:
            lam = np.random.beta(0.3, 0.3)
            idx  = torch.randperm(spec.size(0))
            spec = lam * spec + (1 - lam) * spec[idx]
            y_b  = y[idx]
            optimizer.zero_grad()
            pred = model(spec)
            loss = lam * loss_fn(pred, y) + (1 - lam) * loss_fn(pred, y_b)
        else:
            optimizer.zero_grad()
            loss = loss_fn(model(spec), y)

        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)


def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for wav, y in loader:
            wav  = wav.to(DEVICE, non_blocking=True)
            spec = wav_to_spec_gpu(wav)
            preds.append(model(spec).argmax(1).cpu())
            targets.append(y)
    preds   = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    return (preds == targets).mean(), f1_score(targets, preds, average="macro")

def mixup_batch(wav, y, alpha=0.3):
    lam    = np.random.beta(alpha, alpha)
    idx    = torch.randperm(wav.size(0))
    mixed  = lam * wav + (1 - lam) * wav[idx]
    y_a, y_b = y, y[idx]
    return mixed, y_a, y_b, lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)



In [9]:
import gc

print("Loading ALL training audio into RAM once...")
df_full = pd.read_csv(TRAIN_CSV)

all_wavs   = {}   # id -> wav tensor
all_labels = {}   # id -> label

mel_fn = MelSpectrogram(sample_rate=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
db_fn  = AmplitudeToDB()

for _, row in tqdm(df_full.iterrows(), total=len(df_full)):
    path = os.path.join(TRAIN_AUDIO, f"{row.id}.wav")
    data, sr = sf.read(path, dtype='float32')
    if data.ndim > 1:
        data = data.mean(axis=1)
    wav = torch.tensor(data).unsqueeze(0)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    wav = pad_or_crop(wav)
    # Pre-compute spectrogram too
    spec = db_fn(mel_fn(wav))
    spec = (spec - spec.mean()) / (spec.std() + 1e-6)
    all_wavs[row.id]   = wav
    all_labels[row.id] = int(row.label)

print(f"Done! {len(all_wavs)} samples in RAM.")
gc.collect()


Loading ALL training audio into RAM once...


100%|██████████| 37800/37800 [03:27<00:00, 181.90it/s]

Done! 37800 samples in RAM.


0

In [10]:
set_seed(SEED)

df   = pd.read_csv(TRAIN_CSV)
ids  = df["id"].values
lbls = df["label"].values

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(ids, lbls)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold}  |  train={len(train_idx)}  val={len(val_idx)}")
    print(f"{'='*50}")

    train_ids = ids[train_idx]
    val_ids   = ids[val_idx]

    train_ds = DigitDataset(train_ids, train=True)
    val_ds   = DigitDataset(val_ids,   train=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0, pin_memory=True)

    model     = DigitNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-5
    )
    loss_fn   = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_f1   = 0.0
    save_path = os.path.join(OUTPUT_DIR, f"best_model_fold{fold}.pt")

    for epoch in range(EPOCHS):
        loss           = train_epoch(model, train_loader, optimizer, loss_fn)
        val_acc, val_f1 = evaluate(model, val_loader)
        scheduler.step()

        print(f"Fold {fold} | Epoch {epoch+1:02d}/{EPOCHS} | "
              f"Loss {loss:.4f} | ValAcc {val_acc:.4f} | ValF1 {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), save_path)

    print(f"\nFold {fold} Best Val F1: {best_f1:.4f}")
    fold_results.append(best_f1)

print(f"\nAll folds done! Mean F1: {np.mean(fold_results):.4f}")



FOLD 0  |  train=30240  val=7560
Fold 0 | Epoch 01/50 | Loss 1.9733 | ValAcc 0.8437 | ValF1 0.8404
Fold 0 | Epoch 02/50 | Loss 1.3468 | ValAcc 0.9627 | ValF1 0.9626
Fold 0 | Epoch 03/50 | Loss 1.2192 | ValAcc 0.9683 | ValF1 0.9682
Fold 0 | Epoch 04/50 | Loss 1.1390 | ValAcc 0.9753 | ValF1 0.9753
Fold 0 | Epoch 05/50 | Loss 1.0761 | ValAcc 0.9868 | ValF1 0.9867
Fold 0 | Epoch 06/50 | Loss 1.1309 | ValAcc 0.9862 | ValF1 0.9862
Fold 0 | Epoch 07/50 | Loss 1.0514 | ValAcc 0.9876 | ValF1 0.9875
Fold 0 | Epoch 08/50 | Loss 1.0353 | ValAcc 0.9899 | ValF1 0.9899
Fold 0 | Epoch 09/50 | Loss 1.0309 | ValAcc 0.9907 | ValF1 0.9907
Fold 0 | Epoch 10/50 | Loss 1.0403 | ValAcc 0.9910 | ValF1 0.9910
Fold 0 | Epoch 11/50 | Loss 1.0813 | ValAcc 0.9779 | ValF1 0.9779
Fold 0 | Epoch 12/50 | Loss 1.0601 | ValAcc 0.9819 | ValF1 0.9818
Fold 0 | Epoch 13/50 | Loss 1.0727 | ValAcc 0.9881 | ValF1 0.9881
Fold 0 | Epoch 14/50 | Loss 1.0496 | ValAcc 0.9860 | ValF1 0.9860
Fold 0 | Epoch 15/50 | Loss 1.0578 | ValAc

In [11]:
import gc

print("Loading ALL training audio into RAM once...")
df_full = pd.read_csv(TRAIN_CSV)

all_wavs   = {}
all_labels = {}

for _, row in tqdm(df_full.iterrows(), total=len(df_full)):
    path = os.path.join(TRAIN_AUDIO, f"{row.id}.wav")
    data, sr = sf.read(path, dtype='float32')
    if data.ndim > 1:
        data = data.mean(axis=1)
    wav = torch.tensor(data).unsqueeze(0)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    wav = pad_or_crop(wav)
    all_wavs[row.id]   = wav
    all_labels[row.id] = int(row.label)

print(f"Done! {len(all_wavs)} train samples in RAM.")
gc.collect()


Loading ALL training audio into RAM once...


100%|██████████| 37800/37800 [02:45<00:00, 228.60it/s]

Done! 37800 train samples in RAM.


0

In [12]:
print("Loading ALL test audio into RAM...")
test_df = pd.read_csv(TEST_CSV)
all_test_wavs = {}

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    path = os.path.join(TEST_AUDIO, f"{row.id}.wav")
    data, sr = sf.read(path, dtype='float32')
    if data.ndim > 1:
        data = data.mean(axis=1)
    wav = torch.tensor(data).unsqueeze(0)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    wav = pad_or_crop(wav)
    all_test_wavs[row.id] = wav

print(f"Done! {len(all_test_wavs)} test samples in RAM.")
gc.collect()


Loading ALL test audio into RAM...


100%|██████████| 16200/16200 [03:47<00:00, 71.34it/s]


Done! 16200 test samples in RAM.


0

In [13]:
# ── Test Dataset ─────────────────────────────────────────────
class TestDataset(Dataset):
    def __init__(self, ids, shift=0):
        self.ids   = ids
        self.shift = shift

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        aid = self.ids[idx]
        wav = all_test_wavs[aid].clone()
        if self.shift != 0:
            wav = torch.roll(wav, shifts=self.shift, dims=-1)
            if self.shift > 0:
                wav[..., :self.shift] = 0
            else:
                wav[..., self.shift:] = 0
        return wav

# ── Load fold models ──────────────────────────────────────────
models = []
for fold in range(N_FOLDS):
    ckpt = os.path.join(OUTPUT_DIR, f"best_model_fold{fold}.pt")
    if not os.path.exists(ckpt):
        print(f"WARNING: fold {fold} not found!")
        continue
    m = DigitNet().to(DEVICE)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    m.eval()
    models.append(m)
    print(f"Loaded fold {fold}")

print(f"Total models loaded: {len(models)}")

# ── Inference with TTA ────────────────────────────────────────
test_ids   = test_df["id"].values
all_probs  = []

for shift in TTA_SHIFTS:
    print(f"Running TTA shift={shift}...")
    ds     = TestDataset(test_ids, shift=shift)
    loader = DataLoader(ds, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=4,
                        pin_memory=True)
    for model in models:
        probs = []
        with torch.no_grad():
            for wav in loader:
                wav  = wav.to(DEVICE, non_blocking=True)
                spec = wav_to_spec_gpu(wav)
                prob = torch.softmax(model(spec), dim=1).cpu().numpy()
                probs.append(prob)
        all_probs.append(np.concatenate(probs, axis=0))

# ── Average and save ──────────────────────────────────────────
avg_probs  = np.mean(all_probs, axis=0)
preds      = avg_probs.argmax(axis=1)

submission = pd.DataFrame({"id": test_ids, "label": preds})
sub_path   = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(sub_path, index=False)

print(f"\nSaved submission.csv → {sub_path}")
print(f"Total predictions: {len(submission)}")
print("\nLabel distribution:")
print(submission["label"].value_counts().sort_index())


Loaded fold 0
Loaded fold 1
Loaded fold 2
Loaded fold 3
Loaded fold 4
Total models loaded: 5
Running TTA shift=0...
Running TTA shift=800...
Running TTA shift=-800...
Running TTA shift=400...
Running TTA shift=-400...
Running TTA shift=1200...
Running TTA shift=-1200...
Running TTA shift=200...
Running TTA shift=-200...
Running TTA shift=600...
Running TTA shift=-600...

Saved submission.csv → /kaggle/working/submission.csv
Total predictions: 16200

Label distribution:
label
0    1574
1    1684
2    1628
3    1581
4    1602
5    1635
6    1592
7    1628
8    1640
9    1636
Name: count, dtype: int64


<!-- spectrogram ON the GPU  -->
<!-- F1 score  -->